# 02 — Descarga histórica de precios de carburantes

**Proyecto:** RepostaPro — Optimización del repostaje en flotas comerciales  
**Autor:** Víctor González Martín  
**Notebook:** 02 — Descarga sistemática del histórico desde la API REST

## Objetivo del notebook

Construir la serie temporal histórica de precios de carburantes en España mediante consultas iteradas al endpoint histórico de la API REST del Ministerio para la Transición Ecológica y el Reto Demográfico.

## Parámetros de la descarga

| Parámetro | Valor |
|---|---|
| Fecha inicio | 1 de enero de 2024 |
| Fecha fin | 14 de junio de 2026 (día anterior a la ejecución) |
| Número de días | ~895 días |
| Frecuencia | Una toma diaria |

## Salida esperada

- **Carpeta `data/raw/`**: un fichero JSON por día (~10 GB en total, no se sube a Git).
- **Fichero `logs/descarga_<fecha>.log`**: registro detallado del proceso.
- **Resumen final**: número de descargas exitosas, fallidas y omitidas.

## Importación del módulo de descarga

El módulo `src/descarga.py` se importa como librería. Como el notebook está en `notebooks/` y el módulo en `src/`, añadimos la raíz del proyecto al `sys.path` para poder importarlo.

In [2]:
# Añadir la raíz del proyecto al sys.path para importar desde src/
from pathlib import Path
import sys

RAIZ_PROYECTO = Path("..").resolve()
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

print(f"Raíz del proyecto: {RAIZ_PROYECTO}")

# Importamos las funciones del módulo de descarga
from src.descarga import descargar_rango, descargar_una_fecha, configurar_logger

print("Módulo src.descarga importado correctamente")

Raíz del proyecto: C:\TFM
Módulo src.descarga importado correctamente


## Descarga de una única fecha de prueba

Antes de lanzar la descarga masiva, validamos que la lógica funciona correctamente con una única fecha de prueba. Esto nos permite detectar problemas (errores de red, problemas de autenticación, formato inesperado) sin gastar 45 minutos de descarga completa.

Probamos con el **1 de enero de 2024**, la primera fecha del rango previsto.

In [3]:
# Prueba con una única fecha
from datetime import date
from pathlib import Path
from src.descarga import descargar_una_fecha, configurar_logger

# Configuración de rutas para la prueba
carpeta_raw = Path("../data/raw")
ruta_log_prueba = Path("../logs/prueba_humo.log")

# Aseguramos que las carpetas existen
carpeta_raw.mkdir(parents=True, exist_ok=True)
ruta_log_prueba.parent.mkdir(parents=True, exist_ok=True)

# Configuramos un logger temporal para la prueba
logger_prueba = configurar_logger(ruta_log_prueba)

# Lanzamos la descarga de una sola fecha
fecha_prueba = date(2024, 1, 1)
ruta_salida_prueba = carpeta_raw / f"precios_{fecha_prueba.isoformat()}.json"

print(f"Probando con fecha: {fecha_prueba}")
print(f"Salida prevista:    {ruta_salida_prueba}")
print()

exito = descargar_una_fecha(fecha_prueba, ruta_salida_prueba, logger_prueba)

print()
if exito:
    print(f" Prueba EXITOSA. Fichero descargado: {ruta_salida_prueba}")
    print(f" Tamaño: {ruta_salida_prueba.stat().st_size / 1024 / 1024:.2f} MB")
else:
    print("Prueba FALLIDA. Revisar el log para identificar el problema.")

Probando con fecha: 2024-01-01
Salida prevista:    ..\data\raw\precios_2024-01-01.json



2026-06-22 21:17:55 [INFO]   OK 2024-01-01 → 10943 estaciones (12.1 MB)



 Prueba EXITOSA. Fichero descargado: ..\data\raw\precios_2024-01-01.json
 Tamaño: 12.08 MB


## Descarga histórica completa

Si la prueba de humo ha sido exitosa, lanzamos la descarga del rango completo.

**Importante**: la descarga puede tardar entre 30 y 60 minutos. Durante este tiempo, mantén la terminal abierta y comprueba el progreso por el log o por la salida de la celda. Si necesitas interrumpir la ejecución, puedes hacerlo (los ficheros ya descargados se conservan) y reanudar la descarga en una sesión posterior simplemente volviendo a ejecutar la celda: el módulo detecta automáticamente los ficheros ya existentes y los omite.

In [4]:
# DESCARGA HISTÓRICA COMPLETA
from datetime import date
from pathlib import Path
from src.descarga import descargar_rango

# Configuración del rango de fechas
FECHA_INICIO = date(2024, 1, 1)
FECHA_FIN = date(2026, 6, 14)  # día anterior a la ejecución

# Rutas de destino
carpeta_raw = Path("../data/raw")
ruta_log = Path(f"../logs/descarga_{date.today().isoformat()}.log")

print("=" * 70)
print("LANZANDO DESCARGA HISTÓRICA COMPLETA")
print("=" * 70)
print(f"Desde:   {FECHA_INICIO}")
print(f"Hasta:   {FECHA_FIN}")
print(f"Días:    {(FECHA_FIN - FECHA_INICIO).days + 1}")
print(f"Destino: {carpeta_raw.resolve()}")
print(f"Log:     {ruta_log.resolve()}")
print()
print("La descarga comenzará en breve. Esto puede tardar 30-60 minutos.")
print("Puedes interrumpir con el botón ■ (Stop) y reanudar más adelante.")
print("=" * 70)

# Lanzamos la descarga
resultado = descargar_rango(fecha_inicio=FECHA_INICIO,fecha_fin=FECHA_FIN,carpeta_raw=carpeta_raw,ruta_log=ruta_log,)

2026-06-22 21:18:59 [INFO] ======================================================================
2026-06-22 21:18:59 [INFO] INICIO DE DESCARGA HISTÓRICA
2026-06-22 21:18:59 [INFO] Rango: 2024-01-01 → 2026-06-14 (896 fechas)
2026-06-22 21:18:59 [INFO] Carpeta destino: ..\data\raw
2026-06-22 21:18:59 [INFO] ======================================================================
2026-06-22 21:18:59 [INFO] [1/896] OMITIDO (ya existe): 2024-01-01
2026-06-22 21:18:59 [INFO] [2/896] OMITIDO (ya existe): 2024-01-02
2026-06-22 21:18:59 [INFO] [3/896] OMITIDO (ya existe): 2024-01-03
2026-06-22 21:18:59 [INFO] [4/896] OMITIDO (ya existe): 2024-01-04
2026-06-22 21:18:59 [INFO] [5/896] OMITIDO (ya existe): 2024-01-05
2026-06-22 21:18:59 [INFO] [6/896] OMITIDO (ya existe): 2024-01-06
2026-06-22 21:18:59 [INFO] [7/896] OMITIDO (ya existe): 2024-01-07
2026-06-22 21:18:59 [INFO] [8/896] OMITIDO (ya existe): 2024-01-08
2026-06-22 21:18:59 [INFO] [9/896] OMITIDO (ya existe): 2024-01-09
2026-06-22 21:18:5

LANZANDO DESCARGA HISTÓRICA COMPLETA
Desde:   2024-01-01
Hasta:   2026-06-14
Días:    896
Destino: C:\TFM\data\raw
Log:     C:\TFM\logs\descarga_2026-06-22.log

La descarga comenzará en breve. Esto puede tardar 30-60 minutos.
Puedes interrumpir con el botón ■ (Stop) y reanudar más adelante.


2026-06-22 21:18:59 [INFO] [101/896] OMITIDO (ya existe): 2024-04-10
2026-06-22 21:18:59 [INFO] [102/896] OMITIDO (ya existe): 2024-04-11
2026-06-22 21:18:59 [INFO] [103/896] OMITIDO (ya existe): 2024-04-12
2026-06-22 21:18:59 [INFO] [104/896] OMITIDO (ya existe): 2024-04-13
2026-06-22 21:18:59 [INFO] [105/896] OMITIDO (ya existe): 2024-04-14
2026-06-22 21:18:59 [INFO] [106/896] OMITIDO (ya existe): 2024-04-15
2026-06-22 21:18:59 [INFO] [107/896] OMITIDO (ya existe): 2024-04-16
2026-06-22 21:18:59 [INFO] [108/896] OMITIDO (ya existe): 2024-04-17
2026-06-22 21:18:59 [INFO] [109/896] OMITIDO (ya existe): 2024-04-18
2026-06-22 21:18:59 [INFO] [110/896] OMITIDO (ya existe): 2024-04-19
2026-06-22 21:18:59 [INFO] [111/896] OMITIDO (ya existe): 2024-04-20
2026-06-22 21:18:59 [INFO] [112/896] OMITIDO (ya existe): 2024-04-21
2026-06-22 21:18:59 [INFO] [113/896] OMITIDO (ya existe): 2024-04-22
2026-06-22 21:18:59 [INFO] [114/896] OMITIDO (ya existe): 2024-04-23
2026-06-22 21:18:59 [INFO] [115/89

## Resumen de la descarga

La celda anterior devuelve un diccionario `resultado` con el resumen del proceso. A continuación se inspecciona el contenido para verificar el éxito de la operación.

In [5]:
# Resumen del resultado de la descarga
print("=" * 70)
print("RESUMEN DE LA DESCARGA")
print("=" * 70)
print(f"Total de fechas procesadas: {resultado['total']:,}")
print(f"  Exitosas:                 {resultado['exitosas']:,}")
print(f"  Omitidas (ya existían):   {resultado['omitidas']:,}")
print(f"  Fallidas:                 {resultado['fallidas']:,}")
print(f"\nDuración total: {resultado['duracion']}")
print()

tasa_exito = (resultado["exitosas"] + resultado["omitidas"]) / resultado["total"] * 100
print(f"Tasa de éxito: {tasa_exito:.2f}%")

if resultado["fallidas"] > 0:
    print(f"\n⚠ Quedan {resultado['fallidas']} fechas pendientes. "
          f"Revisar el log en {ruta_log} y volver a ejecutar la celda anterior "
          "para reintentar (las exitosas se omitirán).")
else:
    print("\nDescarga completa. Todos los ficheros disponibles en data/raw/.")

# Conteo real de ficheros en disco
ficheros_descargados = list(carpeta_raw.glob("precios_*.json"))
print(f"\nFicheros JSON en data/raw/: {len(ficheros_descargados):,}")

# Tamaño total ocupado
tamano_total_mb = sum(f.stat().st_size for f in ficheros_descargados) / 1024 / 1024
print(f"Tamaño total: {tamano_total_mb:,.0f} MB ({tamano_total_mb/1024:.2f} GB)")

RESUMEN DE LA DESCARGA
Total de fechas procesadas: 896
  Exitosas:                 3
  Omitidas (ya existían):   893
  Fallidas:                 0

Duración total: 0:00:07.684572

Tasa de éxito: 100.00%

Descarga completa. Todos los ficheros disponibles en data/raw/.

Ficheros JSON en data/raw/: 897
Tamaño total: 11,096 MB (10.84 GB)
